In [ ]:
from __future__ import annotations

# ---------------------------------------------------------------------
# IMPORTANT: cap BLAS thread count BEFORE numpy / sklearn are imported.
# When the GA training loop runs through joblib's loky backend (one
# process per seed), each worker still gets its own BLAS thread pool;
# without this cap, BLAS would oversubscribe cores and tank the speedup.
# os.environ.setdefault preserves any externally-set override.
# ---------------------------------------------------------------------
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS",      "1")
os.environ.setdefault("OMP_NUM_THREADS",      "1")
os.environ.setdefault("BLIS_NUM_THREADS",     "1")

from deap import creator, base
# Define the DEAP creator classes in THIS (parent) process so worker
# results -- which contain creator.Individual / creator.IndividualSingle
# instances -- can be unpickled here. Each worker also recreates them
# lazily inside MultiObjectiveTraining.run / SingleObjectiveTraining.run
# (those guards are idempotent: `if name not in creator.__dict__`).
if "FitnessMulti" not in creator.__dict__:
    creator.create("FitnessMulti", base.Fitness, weights=(1.0, 1.0))
if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMulti)
if "FitnessSingle" not in creator.__dict__:
    creator.create("FitnessSingle", base.Fitness, weights=(1.0,))
if "IndividualSingle" not in creator.__dict__:
    creator.create("IndividualSingle", list, fitness=creator.FitnessSingle)

import numpy
import random
import pandas
import time
from typing import Any
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, f1_score

from training_config import TrainingConfig
from training_utils import select_pareto_individual
from plot_utils import (
    plot_noise_comparison,
    plot_2d_heatmap_grid,
    plot_feature_count_comparison,
    plot_specificity_sensitivity_curve,
)
from multi_objective_training import MultiObjectiveTraining
from single_objective_training import SingleObjectiveTraining
from forward_stepwise_training import ForwardStepwiseTraining
from all_features_training import AllFeaturesTraining
from training_utils import ensure_directory as _ensure_directory
from evaluation_utils import (
    get_continuous_columns, get_dummy_columns, build_model_package
)
from evaluation_utils import (
    apply_proportional_noise, apply_dummy_noise, evaluate_model,
    find_balanced_threshold, compute_aurs,
)

In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    numpy.random.seed(seed)

In [ ]:
#CSV_TRAIN_PATH: str = "arrhythmia/arrhythmia_preprocessed_train_data.csv"
#CSV_TEST_PATH: str = "arrhythmia/arrhythmia_preprocessed_test_data.csv"
#TARGET_COLUMN: str = "Outcome"

CSV_TRAIN_PATH: str = "readmit/readmit_130_hospitals_preprocessed_train_data.csv"
CSV_TEST_PATH: str = "readmit/readmit_130_hospitals_preprocessed_test_data.csv"
TARGET_COLUMN: str = "target_readmitted"

RESULT_PATH: str = time.strftime("%Y-%m-%d_%H-%M-%S")
RESULT_PATH_MULTI: str = f"{RESULT_PATH}\\multi"
RESULT_PATH_SINGLE: str = f"{RESULT_PATH}\\single"
RESULT_PATH_EVAL: str = f"{RESULT_PATH}\\evaluation"

# MORSE pareto front model selection based on knee point algorithm or select the best sign consistency score.
USE_KNEE_POINT_SELECTION: bool = True

# Using ROC-AUC as a main objective function or use PR-AUC instead.
USE_ROC_AUC: bool = True
MAIN_OBJECTIVE: str = "ROC-AUC" if USE_ROC_AUC == True else "PR-AUC"

# Parallelism for the GA training loop.
#   -1 = use all available CPU cores (one worker process per seed, up to that many in flight).
N_JOBS: int = -1

In [ ]:
# Load train dataset.
df_train: pandas.DataFrame = pandas.read_csv(CSV_TRAIN_PATH)

# Split data into training and validation sets.
X_search_pandas: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_search_pandas: pandas.Series = df_train[TARGET_COLUMN]

X_search: numpy.ndarray = numpy.ascontiguousarray(X_search_pandas.to_numpy(), dtype=numpy.float64)
y_search: numpy.ndarray = numpy.ascontiguousarray(y_search_pandas.to_numpy(), dtype=numpy.float64)

# Store feature names.
feature_names: list[str] = list(X_search_pandas.columns)

In [ ]:
# Load test dataset.
df_test: pandas.DataFrame = pandas.read_csv(CSV_TEST_PATH)

y_test: pandas.Series = df_test[TARGET_COLUMN]
X_test: pandas.DataFrame = df_test.drop(columns=[TARGET_COLUMN])

In [ ]:
# Cross-validation splitter shared by all training methods.
# Per-fold marginal correlations for the MORSE sign-consistency
# objective are now computed inside MultiObjectiveTraining itself
# (from each fold's training partition), so no notebook-level
# `corr_array` is needed here.
cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
# Scale the train dataset.
scaler: StandardScaler = StandardScaler()
X_search: numpy.ndarray = scaler.fit_transform(X_search)

## Check features

In [ ]:
# Detect column types automatically from the training DataFrame and
# precompute the per-column training standard deviation used by
# apply_proportional_noise in the noise-robustness sweeps below.
X_train_df: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_train_series: pandas.Series = df_train[TARGET_COLUMN]

continuous_cols: list[str] = get_continuous_columns(X_train_df)
dummy_cols: list[str] = get_dummy_columns(X_train_df)

# Per-column standard deviation on the training set; used by
# apply_proportional_noise to scale Gaussian noise and mean-shift by
# each feature's natural spread.
train_std: pandas.Series = X_train_df.std()

print(f"Continuous features: {len(continuous_cols)}")
print(f"Dummy features:     {len(dummy_cols)}")

In [ ]:
training_results_multi: dict[int, list[list[creator.Individual]]] = {}
training_results_single: dict[int, list[creator.IndividualSingle]] = {}
training_results_fwd: dict[int, list[int]] = {}
training_results_all: dict[int, list[int]] = {}

# 20 seeds fit training, deterministic ordering keeps results comparable across runs.
seeds: list[int] = list(range(42, 62))

# Inner SFS parallelism: 1 when the outer seed pool is parallel
# (avoid loky x sklearn oversubscription); -1 when running sequentially.
_sfs_inner_n_jobs: int = 1 if N_JOBS != 1 else -1

def _train_one_seed(s: int):
    """
    Run MORSE + SOGA + SFS + AllFeatures end-to-end for one seed and return the results.
    """
    _seed_start: float = time.time()
    print(f"  [seed {s:>4d}] starting MORSE + SOGA + SFS + AllFeatures pipeline...", flush=True)

    # --- MORSE ---
    set_seed(s)
    mo: MultiObjectiveTraining = MultiObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_MULTI, use_roc_auc=USE_ROC_AUC),
        feature_names=feature_names,
        X_train=X_search,
        y_train=y_search,
        cv=cv,
    )
    pareto_front: list[creator.Individual] = mo.run()
    mo.clear_cache()
    _morse_dt: float = time.time() - _seed_start
    print(f"  [seed {s:>4d}] MORSE done after {_morse_dt/60:5.1f} min "
          f"({len(pareto_front)} Pareto inds)", flush=True)

    # --- SOGA ---
    set_seed(s)
    _soga_start: float = time.time()
    so: SingleObjectiveTraining = SingleObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_SINGLE, use_roc_auc=USE_ROC_AUC),
        feature_names=feature_names,
        X_train=X_search,
        y_train=y_search,
        cv=cv,
    )
    best_indi: creator.IndividualSingle = so.run()
    so.clear_cache()
    _soga_dt: float = time.time() - _soga_start
    print(f"  [seed {s:>4d}] SOGA done after {_soga_dt/60:5.1f} min", flush=True)

    # --- SFS ---
    set_seed(s)
    _sfs_start: float = time.time()
    sfs: ForwardStepwiseTraining = ForwardStepwiseTraining(
        config=TrainingConfig(seed=s, result_directory="", use_roc_auc=USE_ROC_AUC),
        X_train=X_search,
        y_train=y_search,
        cv=cv,
        inner_n_jobs=_sfs_inner_n_jobs,
    )
    fwd_individual: list[int] = sfs.run()
    _sfs_dt: float = time.time() - _sfs_start
    print(f"  [seed {s:>4d}] SFS done after {_sfs_dt/60:5.1f} min", flush=True)

    # --- All-features (no selection) ---
    all_features_indi: list[int] = AllFeaturesTraining(
        config=TrainingConfig(seed=s, result_directory="", use_roc_auc=USE_ROC_AUC),
        feature_names=feature_names,
    ).run()

    _total_dt: float = time.time() - _seed_start
    print(f"  [seed {s:>4d}] all four methods done -- seed total {_total_dt/60:5.1f} min",
          flush=True)

    return s, pareto_front, best_indi, fwd_individual, all_features_indi


# Seed-level parallelism. Each worker process runs one full
# (MORSE + SOGA + SFS + AllFeatures) pipeline for a single seed.
_overall_start: float = time.time()
print(f"Training {len(seeds)} seeds in parallel (N_JOBS={N_JOBS}, backend=loky)...",
      flush=True)

_results: list = []
if N_JOBS == 1:
    for _i, _s in enumerate(seeds, 1):
        _results.append(_train_one_seed(_s))
        _el: float = time.time() - _overall_start
        print(f"Progress: {_i}/{len(seeds)} seeds done | "
              f"elapsed {_el/60:6.1f} min | avg {_el/_i/60:5.1f} min/seed",
              flush=True)
else:
    _parallel = Parallel(n_jobs=N_JOBS, backend="loky", return_as="generator")
    _gen = _parallel(delayed(_train_one_seed)(s) for s in seeds)
    for _i, _r in enumerate(_gen, 1):
        _results.append(_r)
        _el = time.time() - _overall_start
        _eta = _el / _i * (len(seeds) - _i)
        print(f"Progress: {_i}/{len(seeds)} seeds done | "
              f"elapsed {_el/60:6.1f} min | avg {_el/_i/60:5.1f} min/seed | "
              f"ETA ~{_eta/60:5.1f} min",
              flush=True)

for s, pareto_front, best_indi, fwd_individual, all_features_indi in _results:
    training_results_multi[s] = pareto_front
    training_results_single[s] = best_indi
    training_results_fwd[s] = fwd_individual
    training_results_all[s] = all_features_indi

_total_min: float = (time.time() - _overall_start) / 60.0
print(f"All {len(seeds)} seeds finished in {_total_min:.1f} minutes.", flush=True)

## All-Models Noise Robustness Comparison (averaged across seeds)

Evaluate all four models on the **test set** under a 2D Gaussian sweep plus a separate corruption sweep:
- **Gaussian noise** on continuous features (zero mean-shift), noise level = 0.0 .. 1.0 in 0.1 steps.
- **Covariate shift** on continuous features at a **fixed Gaussian noise level (0.3)**, mean-shift = -1.0 .. 1.0 in 0.2 steps.
- **Random corruption** on binary (dummy) features, corruption fraction = 0.0 .. 1.0 in 0.1 steps.

The Gaussian-noise and covariate-shift line plots are both 1D slices of the same 2D (noise_level x mean_shift) sweep, so no evaluations are duplicated between them and the heatmap grid below.

For each seed the same four models are evaluated at every noise level, and the mean AUC across seeds is plotted with an std shaded band. This gives a like-for-like view of robustness for:
- **Multi-Objective (MORSE)** Pareto individual with max sign-consistency or knee point
- **Single-Objective (AUC-only GA)** best HoF individual
- **All features (no selection)** logistic regression on every input
- **Forward stepwise selection** `SequentialFeatureSelector` (forward, CV=3, PR-AUC or ROC-AUC, tol=1e-3)

The three line-plot outputs (`gaussian_noise_comparison_test.png`, `covariate_shift_comparison_test.png`, `dummy_flip_comparison_test.png`) share the same style, with four curves and seed-aggregated statistics.

The 2D heatmap grid (`gaussian_2d_heatmap_grid_test.png`) additionally reports each method's **AURS** (Area Under the Robustness Surface) in its panel subtitle: the average fraction of that SAME method's own clean-test score it retains across the whole (noise_level x mean_shift) grid, so methods with different clean-test baselines are compared purely on how gracefully they degrade, not on raw score. The full definition and calculation are documented in `evaluation_utils.compute_aurs`; the per-method numbers are also written to `aurs_scores.csv` alongside the plot.

In [ ]:
# All-models noise robustness comparison (averaged across seeds).
#
# One 2D Gaussian sweep (noise_level x mean_shift) and one 1D dummy
# corruption sweep are executed per seed for every model. The Gaussian
# line plot at zero mean-shift, and the covariate-shift line plot at a
# fixed noise level, are both derived from the 2D sweep to avoid
# duplicating evaluations. Plotting goes through plot_utils.
noise_levels: numpy.ndarray = numpy.arange(0.0, 1.1, 0.1)
shift_levels: numpy.ndarray = numpy.arange(-1.0, 1.1, 0.2)
flip_levels: numpy.ndarray = numpy.arange(0.0, 1.05, 0.1)

# Fixed Gaussian noise level used for the covariate-shift line plot below
# (a 1D slice of the 2D sweep at this noise level, varying mean_shift).
FIXED_NOISE_LEVEL_FOR_SHIFT_SWEEP: float = 0.3

heatmap_rows: list[dict] = []
dummy_rows: list[dict] = []

print("Running noise sweeps for all 4 models across all seeds...")
for s in seeds:
    set_seed(s)

    # --- Build all 4 model packages from the cached bit-vectors ---
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = select_pareto_individual(
        pareto_front, use_knee_point=USE_KNEE_POINT_SELECTION)

    pkg_multi: dict[str, Any] = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    pkg_single: dict[str, Any] = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)
    pkg_all: dict[str, Any] = build_model_package(
        training_results_all[s], feature_names, X_train_df, y_train_series, seed=s)
    pkg_fwd: dict[str, Any] = build_model_package(
        training_results_fwd[s], feature_names, X_train_df, y_train_series, seed=s)

    models: dict[str, dict] = {
        "multi":   pkg_multi,
        "single":  pkg_single,
        "all":     pkg_all,
        "forward": pkg_fwd,
    }

    # --- 2D Gaussian sweep: noise_level x mean_shift ---
    for noise in noise_levels:
        nv: float = round(float(noise), 2)
        for shift in shift_levels:
            sv: float = round(float(shift), 2)
            X_noisy: pandas.DataFrame = apply_proportional_noise(
                X_test, train_std, nv, sv, continuous_cols)
            row: dict = {"seed": s, "noise_level": nv, "mean_shift": sv}
            for label, pkg in models.items():
                row[f"auc_{label}"] = evaluate_model(
                    pkg, X_noisy, y_test, use_roc_auc=USE_ROC_AUC)
            heatmap_rows.append(row)

    # --- 1D Dummy corruption sweep ---
    for flip in flip_levels:
        fv: float = round(float(flip), 2)
        X_noisy = apply_dummy_noise(X_test, fv, dummy_cols)
        row = {"seed": s, "flip_rate": fv}
        for label, pkg in models.items():
            row[f"auc_{label}"] = evaluate_model(
                pkg, X_noisy, y_test, use_roc_auc=USE_ROC_AUC)
        dummy_rows.append(row)

    print(f"  Seed {s} done "
          f"(features: multi={sum(best_multi_ind)}, single={sum(training_results_single[s])}, "
          f"all={sum(training_results_all[s])}, forward={sum(training_results_fwd[s])})")

heatmap_df: pandas.DataFrame = pandas.DataFrame(heatmap_rows)
dummy_df: pandas.DataFrame = pandas.DataFrame(dummy_rows)

# 1D Gaussian line = 2D slice at mean_shift == 0.0
gauss_df: pandas.DataFrame = heatmap_df[
    numpy.isclose(heatmap_df["mean_shift"], 0.0)
].drop(columns=["mean_shift"]).reset_index(drop=True)

# 1D covariate-shift line = 2D slice at noise_level == FIXED_NOISE_LEVEL_FOR_SHIFT_SWEEP
shift_df: pandas.DataFrame = heatmap_df[
    numpy.isclose(heatmap_df["noise_level"], FIXED_NOISE_LEVEL_FOR_SHIFT_SWEEP)
].drop(columns=["noise_level"]).reset_index(drop=True)

# Aggregations across seeds.
gauss_agg: pandas.DataFrame = gauss_df.drop(columns=["seed"]).groupby("noise_level").agg(["mean", "std"])
shift_agg: pandas.DataFrame = shift_df.drop(columns=["seed"]).groupby("mean_shift").agg(["mean", "std"])
dummy_agg: pandas.DataFrame = dummy_df.drop(columns=["seed"]).groupby("flip_rate").agg(["mean", "std"])
heatmap_agg: pandas.DataFrame = heatmap_df.drop(columns=["seed"]).groupby(
    ["noise_level", "mean_shift"]).mean()

# Plot style: one entry per model (label, color, marker, linestyle)
model_styles: dict[str, tuple] = {
    "multi":   ("Multi-Objective (MORSE)",       "tab:blue",   "o", "-"),
    "single":  ("Single-Objective (AUC-only GA)", "tab:orange", "s", "--"),
    "all":     ("All features (no selection)",    "tab:green",  "^", "-."),
    "forward": ("Forward stepwise selection",     "tab:red",    "D", ":"),
}

# AURS (Area Under the Robustness Surface): collapse each model's whole 2D
# noise x shift sweep into a single score -- the average fraction of that
# SAME model's own clean-test score it retains across the grid. This is
# deliberately relative to each model's own clean cell (not a raw AUC
# average) so models with different clean-test baselines are compared on
# degradation alone. See evaluation_utils.compute_aurs for the full
# definition and the exact calculation (retention-ratio normalisation,
# then double trapezoidal integration over the grid, renormalised by the
# grid's area).
aurs_scores: dict[str, float] = {
    key: compute_aurs(heatmap_agg, key) for key in model_styles
}

comparison_dir: str = os.path.join(RESULT_PATH_EVAL, "all_models_comparison")
_ensure_directory(comparison_dir)
heatmap_df.to_csv(os.path.join(comparison_dir, "gaussian_2d_per_seed.csv"), index=False)
gauss_df.to_csv(os.path.join(comparison_dir, "gaussian_noise_per_seed.csv"), index=False)
shift_df.to_csv(os.path.join(comparison_dir, "covariate_shift_per_seed.csv"), index=False)
dummy_df.to_csv(os.path.join(comparison_dir, "dummy_flip_per_seed.csv"), index=False)

aurs_df: pandas.DataFrame = pandas.DataFrame([
    {"model_key": key, "method": model_styles[key][0], "aurs": aurs_scores[key]}
    for key in model_styles
])
aurs_df.to_csv(os.path.join(comparison_dir, "aurs_scores.csv"), index=False)

print("\nAURS (Area Under the Robustness Surface) per model -- avg. fraction of "
      f"clean-test {MAIN_OBJECTIVE} retained over noise_level in "
      f"[{noise_levels.min():.1f}, {noise_levels.max():.1f}] x mean_shift in "
      f"[{shift_levels.min():.1f}, {shift_levels.max():.1f}]:")
for key in model_styles:
    print(f"  {model_styles[key][0]:35s} AURS = {aurs_scores[key]:.1%}")

plot_noise_comparison(
    agg_df=gauss_agg,
    model_styles=model_styles,
    xlabel="Gaussian Noise Level (fraction of training std, no mean shift)",
    ylabel=f"Test {MAIN_OBJECTIVE} (mean across seeds; shaded = +/- 1 std)",
    title=("Robustness on Test Set: Additive Gaussian Noise on Continuous Features\n"
           "(all 4 models, averaged across seeds)"),
    out_path=os.path.join(comparison_dir, "gaussian_noise_comparison_test.png"),
)

plot_noise_comparison(
    agg_df=shift_agg,
    model_styles=model_styles,
    xlabel=(f"Covariate Shift (fraction of training std; fixed Gaussian noise "
            f"level = {FIXED_NOISE_LEVEL_FOR_SHIFT_SWEEP})"),
    ylabel=f"Test {MAIN_OBJECTIVE} (mean across seeds; shaded = +/- 1 std)",
    title=("Robustness on Test Set: Covariate Shift on Continuous Features\n"
           f"(fixed Gaussian noise level = {FIXED_NOISE_LEVEL_FOR_SHIFT_SWEEP}, "
           "all 4 models, averaged across seeds)"),
    out_path=os.path.join(comparison_dir, "covariate_shift_comparison_test.png"),
)

plot_noise_comparison(
    agg_df=dummy_agg,
    model_styles=model_styles,
    xlabel="Corruption Fraction (proportion of dummy cells randomised)",
    ylabel=f"Test {MAIN_OBJECTIVE} (mean across seeds; shaded = +/- 1 std)",
    title=("Robustness on Test Set: Random Corruption Noise on Binary (Dummy) Features\n"
           "(all 4 models, averaged across seeds)"),
    out_path=os.path.join(comparison_dir, "dummy_flip_comparison_test.png"),
)

plot_2d_heatmap_grid(
    heatmap_agg=heatmap_agg,
    model_styles=model_styles,
    cbar_label=f"Test {MAIN_OBJECTIVE}",
    xlabel="Gaussian Noise Level (fraction of training std)",
    ylabel="Mean Shift (fraction of training std)",
    title=("Robustness on Test Set: 2D Gaussian sweep (noise x covariate shift)\n"
           f"per-cell mean {MAIN_OBJECTIVE} across seeds; shared colour scale for direct comparison\n"
           "per-panel AURS = avg. fraction of that model's own clean-test score "
           "retained across the whole grid (see evaluation_utils.compute_aurs)"),
    out_path=os.path.join(comparison_dir, "gaussian_2d_heatmap_grid_test.png"),
    aurs_scores=aurs_scores,
)

print(f"\nAll-models comparison plots and CSVs saved to: {comparison_dir}")

## Feature Count Comparison Across Seeds

For each method (MORSE, SOGA, All features, Forward stepwise) plot the number of selected features per seed.

- MORSE and SOGA vary with the random seed (different GA inits / CV shuffles produce different Pareto fronts and HoFs).
- **All features** is by construction constant at `n_features`.
- **Forward stepwise** selects its own number of features per seed via `SequentialFeatureSelector`'s tol-based auto stopping.

The plot is a boxplot (quartile summary) with individual seeds overlaid as jittered dots, so both the distribution shape and the per-seed values are visible. Depends on the four `training_results_*` dictionaries populated by the per-seed training loop.

In [ ]:
# Feature count comparison across seeds.
#
# Reads directly from the four training_results_* caches populated by
# `_train_one_seed`. MORSE's per-seed bit-vector is resolved through
# `select_pareto_individual` using the same USE_KNEE_POINT_SELECTION
# flag as the downstream evaluation cell. The actual boxplot +
# stripplot rendering is delegated to plot_utils.
feature_count_rows: list[dict] = []
for s in seeds:
    _multi_ind = select_pareto_individual(
        training_results_multi[s], use_knee_point=USE_KNEE_POINT_SELECTION)
    feature_count_rows.append({
        "method":     "MORSE",
        "seed":       s,
        "n_features": int(sum(_multi_ind)),
    })
    feature_count_rows.append({
        "method":     "SOGA (AUC)",
        "seed":       s,
        "n_features": int(sum(training_results_single[s])),
    })
    feature_count_rows.append({
        "method":     "Forward stepwise",
        "seed":       s,
        "n_features": int(sum(training_results_fwd[s])),
    })
    feature_count_rows.append({
        "method":     "All features",
        "seed":       s,
        "n_features": int(sum(training_results_all[s])),
    })

feature_count_df: pandas.DataFrame = pandas.DataFrame(feature_count_rows)

print("=== Feature Count Comparison Across Seeds ===\n")
summary_df: pandas.DataFrame = (
    feature_count_df.groupby("method")["n_features"]
    .agg(["min", "median", "mean", "max", "std"])
    .round(2)
)
print(summary_df.to_string())

method_order: list[str] = ["MORSE", "SOGA (AUC)", "Forward stepwise", "All features"]
palette: dict[str, str] = {
    "MORSE":            "tab:blue",
    "SOGA (AUC)":       "tab:orange",
    "Forward stepwise": "tab:red",
    "All features":     "tab:green",
}

feature_count_dir: str = os.path.join(RESULT_PATH_EVAL, "feature_counts")
_ensure_directory(feature_count_dir)

plot_feature_count_comparison(
    feature_count_df=feature_count_df,
    method_order=method_order,
    palette=palette,
    n_features_total=len(feature_names),
    n_seeds=len(seeds),
    out_path=os.path.join(feature_count_dir, "feature_count_comparison.png"),
)

feature_count_df.to_csv(
    os.path.join(feature_count_dir, "feature_count_per_seed.csv"), index=False)
summary_df.to_csv(
    os.path.join(feature_count_dir, "feature_count_summary.csv"))

print(f"\nFeature count plot and CSVs saved to: {feature_count_dir}")

## Best MORSE Model Across Seeds

For every seed we build the deployed MORSE model (using the same knee-point / max-sign-consistency selection strategy that drives the rest of the notebook) and score it on the test set. The seed whose model achieves the highest test `MAIN_OBJECTIVE` (ROC-AUC or PR-AUC, controlled by `USE_ROC_AUC`) is picked as the **best MORSE model** for downstream reporting.

For that single model we then:
1. compute the classification threshold at which sensitivity ≈ specificity on the test set (`find_balanced_threshold`);
2. apply the threshold to derive binary predictions;
3. report a full metric set at that threshold: accuracy, ROC-AUC, PR-AUC, F1, sensitivity, specificity;
4. save the sensitivity/specificity vs. threshold-rank curve as a PDF next to the other diagnostic artefacts.

In [ ]:
# Best MORSE model across seeds -- deployment-ready evaluation at a balanced
# sensitivity / specificity threshold.

# --- 1) Score every seed's MORSE model on the test set, pick the best -----
_best_seed: int = -1
_best_score: float = -numpy.inf
_best_ind: creator.Individual | None = None

for s in seeds:
    _pareto: list[creator.Individual] = training_results_multi[s]
    _ind: creator.Individual = select_pareto_individual(
        _pareto, use_knee_point=USE_KNEE_POINT_SELECTION)
    _pkg: dict[str, Any] = build_model_package(
        _ind, feature_names, X_train_df, y_train_series, seed=s)
    _test_score: float = evaluate_model(_pkg, X_test, y_test, use_roc_auc=USE_ROC_AUC)
    if _test_score > _best_score:
        _best_score = _test_score
        _best_seed = s
        _best_ind = _ind

# --- 2) Refit the deployed model on the best seed's selection ------------
best_pkg: dict[str, Any] = build_model_package(
    _best_ind, feature_names, X_train_df, y_train_series, seed=_best_seed)
_X_test_scaled: numpy.ndarray = best_pkg["scaler"].transform(
    X_test[best_pkg["features"]].to_numpy())
y_prob_best: numpy.ndarray = best_pkg["model"].predict_proba(_X_test_scaled)[:, 1]

# --- 3) Balanced sensitivity/specificity threshold ------------------------
_balanced: dict[str, Any] = find_balanced_threshold(y_test, y_prob_best)
y_pred_best: numpy.ndarray = _balanced["y_pred"]

# --- 4) Full metric report at the balanced threshold ---------------------
best_metrics: dict[str, Any] = {
    "seed":               _best_seed,
    "n_features":         int(sum(_best_ind)),
    "threshold":          _balanced["threshold"],
    "accuracy":           float(accuracy_score(y_test, y_pred_best)),
    "roc_auc":            float(roc_auc_score(y_test, y_prob_best)),
    "pr_auc":             float(average_precision_score(y_test, y_prob_best)),
    "f1_score":           float(f1_score(y_test, y_pred_best)),
    "sensitivity":        _balanced["sensitivity"],
    "specificity":        _balanced["specificity"],
    "pareto_auc":         float(_best_ind.fitness.values[0]),
    "pareto_sign_score":  float(_best_ind.fitness.values[1]),
}

print(f"=== Best MORSE model across {len(seeds)} seeds ===\n")
print(f"  Selected seed:            {best_metrics['seed']}")
print(f"  Selected features:        {best_metrics['n_features']}")
print(f"  Fitness AUC (from GA):    {best_metrics['pareto_auc']:.4f}")
print(f"  Fitness sign-consistency: {best_metrics['pareto_sign_score']:.4f}")
print(f"  Best test {MAIN_OBJECTIVE}:  {_best_score:.4f}")
print()
print("  --- metrics at the balanced sens/spec threshold ---")
print(f"  Balanced threshold: {best_metrics['threshold']:.4f}")
print(f"  Accuracy:           {best_metrics['accuracy']:.4f}")
print(f"  ROC-AUC:            {best_metrics['roc_auc']:.4f}")
print(f"  PR-AUC:             {best_metrics['pr_auc']:.4f}")
print(f"  F1-score:           {best_metrics['f1_score']:.4f}")
print(f"  Sensitivity:        {best_metrics['sensitivity']:.4f}")
print(f"  Specificity:        {best_metrics['specificity']:.4f}")

# --- 5) Persist metrics + spec/sens curve --------------------------------
best_model_dir: str = os.path.join(RESULT_PATH_EVAL, "best_morse_model")
_ensure_directory(best_model_dir)

pandas.DataFrame([best_metrics]).to_csv(
    os.path.join(best_model_dir, "best_morse_metrics.csv"), index=False)

plot_specificity_sensitivity_curve(
    sorted_scores=_balanced["sorted_scores"],
    sensitivity_curve=_balanced["sensitivity_curve"],
    specificity_curve=_balanced["specificity_curve"],
    intersection_idx=_balanced["intersection_idx"],
    out_path=os.path.join(best_model_dir, "spec_sens_curve.pdf"),
)

print(f"\nBest MORSE model artefacts saved to: {best_model_dir}")